# 02 - Did the data actually land? Freshness, volume, grain and silent loss

A green pipeline run does not prove the rows landed. This notebook checks, per Delta table:

1. **Freshness** - when was the last commit, and is that inside the window the owner expects?
2. **Volume** - is the table empty, or smaller than a floor you set?
3. **Grain** - is the key column unique (no duplicate grain)?
4. **Silent loss** - no null keys, and every child row points at a real parent row (no orphans).

Every check becomes one row (check, target, PASS/FAIL/SKIP, detail, time). The notebook
prints a GREEN or RED verdict and, if anything failed, **raises so the job shows as Failed**.
That is what lets a scheduled run alert you. Optionally it appends the rows to a Delta
table so you keep a platform-health history.

**Before you run it**: fill in the config cell. Table and column names below are examples.
The freshness SLA and the results table are additions made after the demo run and have **not**
been run in a live workspace. The grain, null-key and orphan checks are the same checks that
ran on 2026-11-03 (green, then red with a planted orphan, then green again).


In [ ]:
# ---- settings: change these ----
WORKSPACE_ID = "<your-workspace-id>"
LAKEHOUSE_ID = "<your-lakehouse-id>"

# table -> key column, freshness SLA in hours (None = just report the age), minimum rows (None = no floor)
TABLES = {
    "orders":      {"key": "order_id",      "sla_hours": 26, "min_rows": 1},
    "order_lines": {"key": "order_line_id", "sla_hours": 26, "min_rows": 1},
    "customers":   {"key": "customer_id",   "sla_hours": None, "min_rows": 1},
}
# (child table, child column, parent table, parent column): every child row must have a parent
REFERENCES = [
    ("order_lines", "order_id", "orders", "order_id"),
    ("orders", "customer_id", "customers", "customer_id"),
]

WRITE_RESULTS = False                          # True = append every check to a Delta table (needs a default lakehouse)
RESULTS_TABLE = "observability_check_results"


In [ ]:
# ---- tested helper code (src/check_logic.py) ----
from datetime import datetime, timezone


def make_result(check, target, status, detail):
    """One row of the platform-health history: check name, time, status, detail.

    status is one of PASS, FAIL, SKIP.
    """
    if status not in ("PASS", "FAIL", "SKIP"):
        raise ValueError("status must be PASS, FAIL or SKIP, got %r" % (status,))
    return {
        "check": check,
        "target": target,
        "status": status,
        "detail": detail,
        "checked_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


def freshness_result(table, last_commit_utc, sla_hours, now_utc=None):
    """Is this table inside the window its owner expects?

    last_commit_utc: timezone-aware datetime of the latest Delta commit.
    sla_hours: the maximum age you accept, or None to only report the age.
    """
    now_utc = now_utc or datetime.now(timezone.utc)
    age_h = (now_utc - last_commit_utc).total_seconds() / 3600.0
    if sla_hours is None:
        return make_result("freshness", table, "SKIP", "age %.1f h, no SLA set" % age_h)
    if age_h <= sla_hours:
        return make_result("freshness", table, "PASS", "age %.1f h, SLA %s h" % (age_h, sla_hours))
    return make_result("freshness", table, "FAIL", "age %.1f h is over the %s h SLA" % (age_h, sla_hours))


def count_result(check, target, bad_count, what):
    """A check that passes when it finds zero bad rows (duplicates, orphans, null keys)."""
    if bad_count == 0:
        return make_result(check, target, "PASS", "no %s" % what)
    return make_result(check, target, "FAIL", "%d %s" % (bad_count, what))


def volume_result(table, rows, min_rows):
    """Is the table empty, or smaller than the floor you set?"""
    if min_rows is None:
        return make_result("volume", table, "SKIP", "%d rows, no floor set" % rows)
    if rows >= min_rows:
        return make_result("volume", table, "PASS", "%d rows, floor %d" % (rows, min_rows))
    return make_result("volume", table, "FAIL", "%d rows is under the floor of %d" % (rows, min_rows))


def verdict(results):
    """GREEN when nothing failed, otherwise RED. SKIP never turns a run red."""
    failed = [r for r in results if r["status"] == "FAIL"]
    return ("RED" if failed else "GREEN"), failed


In [ ]:
from datetime import timezone
from delta.tables import DeltaTable
from pyspark.sql import Row, functions as F

# Delta history timestamps come back as naive datetimes in the session time zone.
# Pin the session to UTC so we can safely treat them as UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

BASE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables"
def table_path(t):
    return f"{BASE}/{t}"

frames, results = {}, []
for t, cfg in TABLES.items():
    df = spark.read.format("delta").load(table_path(t))
    frames[t] = df

    last = DeltaTable.forPath(spark, table_path(t)).history(1).select("timestamp").collect()[0]["timestamp"]
    results.append(freshness_result(t, last.replace(tzinfo=timezone.utc), cfg["sla_hours"]))
    results.append(volume_result(t, df.count(), cfg["min_rows"]))

    key = cfg["key"]
    total, distinct = df.count(), df.select(key).distinct().count()
    results.append(count_result("grain", f"{t}.{key}", total - distinct, "duplicate keys"))
    results.append(count_result("null-key", f"{t}.{key}", df.filter(F.col(key).isNull()).count(), "null keys"))

for child, ccol, parent, pcol in REFERENCES:
    orphans = frames[child].join(frames[parent], frames[child][ccol] == frames[parent][pcol], "left_anti").count()
    results.append(count_result("orphan", f"{child}.{ccol} -> {parent}.{pcol}", orphans, "orphan rows"))


In [ ]:
print(f"{'check':10} {'target':38} {'status':6} detail")
for r in results:
    print(f"{r['check']:10} {r['target']:38} {r['status']:6} {r['detail']}")

if WRITE_RESULTS:
    spark.createDataFrame([Row(**r) for r in results]).write.format("delta").mode("append").saveAsTable(RESULTS_TABLE)
    print(f"\nappended {len(results)} rows to {RESULTS_TABLE}")

state, failed = verdict(results)
print("\n" + "=" * 48)
if state == "RED":
    print(f"DATA-TRUST VERDICT: RED ({len(failed)} failed)")
    for r in failed:
        print(f"  - {r['check']} {r['target']}: {r['detail']}")
    raise SystemExit(f"observability checks failed: {[(r['check'], r['target']) for r in failed]}")
print("DATA-TRUST VERDICT: GREEN - data landed at every grain")


## Demo the RED path

Plant an orphan row (a child row whose parent does not exist), re-run, and the verdict flips
to RED and the job fails. Remove the orphan, re-run, and it goes green again. That is the whole
demo: *fresh is not the same as correct*.
